# setup

> The first run: ramabana starts inside its own tmux server, offers to install tmux when it is missing, and `ramabana --doctor` says what it found.

In [ ]:
#| default_exp setup

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq

In [ ]:
#| export
from __future__ import annotations

import hashlib, importlib, importlib.util, json, os, platform, re, shlex, shutil, subprocess, sys, time, uuid

from fastcore.basics import ifnone, patch, store_attr
from fastcore.xtras import Path

## Extras and scripts

This module loads nothing heavy: fastcore and the standard library, never `core`. So the console scripts start here. `need` names the extra a missing package belongs to, and a script whose extra is missing prints that and exits 2. `core` re-exports these names.

In [ ]:
#| export
ENV_DEFAULT = ('RAMABANA_', 'LEELA_')

def env_prefixes():
    "The env prefixes `core` reads: its own once loaded, where an app may have changed them, else `ENV_DEFAULT`."
    return (c.ENV_PREFIX, c.ENV_FALLBACK) if (c := sys.modules.get('ramabana.core')) else ENV_DEFAULT

#: the modules of each extra that ramabana imports
EXTRAS = dict(search=('vishalakshi', 'litesearch', 'kosha', 'rgapi', 'fossick'),
              python=('dhrishti', 'jupyter_client'),
              serve=('mcp', 'acp'),
              cli=('teleprint', 'fastpylight'),
              dhrona=('dhrona',))

def installed(mod):
    "Whether `mod` would import, without importing it."
    try: return importlib.util.find_spec(mod) is not None
    except (ImportError, ValueError): return False

def need(extra, *mods):
    "`''` when `mods`, by default all of `extra`'s, are installed; else what to install."
    miss = [m for m in mods or EXTRAS[extra] if not installed(m)]
    return f"{', '.join(miss)} not installed: pip install 'ramabana[{extra}]'" if miss else ''

#: console scripts besides `run_cli`: the module whose `main` runs, its extra, and the modules it needs
SCRIPTS = dict(run_mcp=('mcp', 'serve', ('mcp',)), run_acp=('racp', 'serve', ('acp',)),
               run_pane=('pane', 'cli', ('teleprint', 'rich')))

def _refuse(msg):
    "A console script that prints `msg` and exits 2."
    def main():
        print(msg, file=sys.stderr)
        return 2
    return main

def __getattr__(name):
    "The `main` a console script in `SCRIPTS` calls itself, so `call_parse` reads the command line; or one naming the missing extra."
    if name not in SCRIPTS: raise AttributeError(f'module {__name__!r} has no attribute {name!r}')
    mod, extra, mods = SCRIPTS[name]
    return _refuse(msg) if (msg := need(extra, *mods)) else importlib.import_module(f'ramabana.{mod}').main

In [ ]:
assert installed('fastcore') and not installed('no_such_module')
test_eq(need('search', 'fastcore'), '')
test_eq(need('search', 'no_such_module'), "no_such_module not installed: pip install 'ramabana[search]'")
test_eq(env_prefixes(), (sys.modules['ramabana.core'].ENV_PREFIX, sys.modules['ramabana.core'].ENV_FALLBACK) if 'ramabana.core' in sys.modules else ENV_DEFAULT)

## The config

The now pane needs a tmux split, and shift+enter needs tmux to pass extended keys through. Each session gets a tmux server of its own, `tmux -L ramabana-<id>`, started with `<cfg>/tmux.conf`. A fresh server reads the current config and inherits the current environment, so a second session never picks up the first one's API keys or virtualenv. That server never reads `~/.tmux.conf`, and your own tmux is left as it was.

`tmux_conf` adds only the lines the installed tmux understands. `extended-keys` arrived in tmux 3.2 and `extended-keys-format csi-u` in 3.5. An older tmux would print an error for each unknown line when the session opens. `RGB` is claimed only when `$COLORTERM` says the terminal draws true colour.

In [ ]:
#| export
SOCKET, WRAP_MIN, SPLIT_MIN, KEYS_MIN = 'ramabana', (3, 0), (3, 1), (3, 2)
TMUX_CONF = r'''# ramabana writes this file and rewrites it when its own copy changes
if-shell 'infocmp tmux-256color' 'set -g default-terminal tmux-256color' 'set -g default-terminal screen-256color'
if-shell '[ "$COLORTERM" = truecolor ] || [ "$COLORTERM" = 24bit ]' "set -as terminal-overrides ',*:RGB'"
set -s escape-time 10
set -g mouse on
set -g status off
set -g remain-on-exit off
set -g destroy-unattached on
'''
VERSIONED = {KEYS_MIN: "set -s extended-keys on\nset -as terminal-features ',*:extkeys'",
             (3, 5): 'set -s extended-keys-format csi-u'}

def tmux_version(s):
    "The `(major, minor)` in `tmux -V` output such as `tmux 3.5a`, or None."
    return tuple(map(int, m.groups())) if (m := re.search(r'(\d+)\.(\d+)', s or '')) else None

def tmux_conf(version):
    "`TMUX_CONF` with the lines tmux `version` understands."
    return TMUX_CONF + ''.join(f'{l}\n' for v, l in VERSIONED.items() if version >= v)

In [ ]:
test_eq([tmux_version(s) for s in ('tmux 3.5a', 'tmux next-3.6', 'tmux master')], [(3, 5), (3, 6), None])
assert 'extended-keys-format csi-u' in tmux_conf((3, 5)) and 'csi-u' not in tmux_conf((3, 4))
assert 'extended-keys on' in tmux_conf((3, 2)) and 'extended-keys' not in tmux_conf((3, 1))
print(tmux_conf((3, 5)))

## The relaunch

`Setup.launch` runs before an interactive session starts. Outside tmux it replaces this process with `tmux new-session` on a new server, running the same command line in the same folder. Quitting ramabana ends the session and returns you to the shell. It stays put for a one-shot question, for `--json`, when stdin or stdout is not a terminal, and when `--tmux off` or `$RAMABANA_TMUX=off` says to. The relaunched process has `$RAMABANA_WRAPPED` set, so it never relaunches again. Inside tmux, `--pane auto` then opens the pane.

`destroy-unattached` ends the session when you detach or close the window, so ramabana never runs on unseen. The history is saved, and `--resume latest` reopens it. The command runs under `sh`, which waits for enter after a non-zero exit, so an error stays on screen instead of vanishing with the session.

Inside a tmux of your own, `launch` turns `extended-keys` on for that server when it is off. The change lasts until the server exits, and only programs that ask for extended keys get them.

Every outside effect goes through a seam: `which`, `system`, `run`, `execvpe`, `ask` and `isatty`, plus `environ`, `argv` and `cwd`.

In [ ]:
#| export
HOLD = r'"$@" || { s=$?; printf "\nramabana exited with status %s · enter closes this\n" $s; read _; exit $s; }'
WHY = {'none': 'no tmux, and no brew, apt-get or dnf to install it', 'no': 'no tmux', '': 'no tmux', 'failed': 'tmux did not install'}

class Setup:
    "The tmux ramabana runs in: its config, the relaunch into it, the install offer, and `--doctor`."
    def __init__(self,
        cfg='~/.config/ramabana', # holds `tmux.conf` and `setup.json`
        environ=None,             # default `os.environ`
        argv=None,                # the command line to rerun; default `sys.argv`
        cwd=None,                 # where the session starts; default the current folder
        which=shutil.which, system=platform.system, run=subprocess.run, execvpe=os.execvpe, ask=input,
        isatty=None,              # whether stdin and stdout are terminals
    ):
        store_attr('which,system,run,execvpe,ask')
        self._about = None
        self.cfg, self.environ, self.argv = Path(cfg or '~/.config/ramabana').expanduser(), ifnone(environ, os.environ), ifnone(argv, sys.argv)
        self.cwd, self.isatty = cwd or os.getcwd(), isatty or (lambda: sys.stdin.isatty() and sys.stdout.isatty())

    def _env(self, name): return next((v for p in env_prefixes() if (v := self.environ.get(p+name))), None)
    @property
    def tmux(self): return self.which('tmux')

    def about(self):
        "`tmux -V`, asked once, or '' without tmux."
        if self._about is None and (t := self.tmux): self._about = self.run([t, '-V'], capture_output=True, text=True).stdout.strip()
        return self._about or ''

    def version(self):
        "The `(major, minor)` of tmux, `(99, 0)` for an unnumbered build, or None without tmux."
        return (tmux_version(a) or (99, 0)) if (a := self.about()) else None

    def conf(self):
        "Write `tmux_conf` for this tmux to `<cfg>/tmux.conf` when it differs, and return the path."
        p, text = self.cfg/'tmux.conf', tmux_conf(self.version() or (0, 0))
        if not p.exists() or p.read_text() != text: p.mk_write(text)
        return p

    def command(self):
        "The argv that reruns this command line: its script, else this Python."
        exe, *args = self.argv
        me = [exe] if os.path.isfile(exe) and os.access(exe, os.X_OK) else [sys.executable, '-c', 'import sys; from ramabana.setup import run_cli; sys.exit(run_cli())']
        return [*me, *args]

    def wrap(self):
        "The `execvpe` arguments that rerun `command` on a tmux server of its own."
        env = {**self.environ, env_prefixes()[0]+'WRAPPED': '1'}
        return self.tmux, ['tmux', '-L', f'{SOCKET}-{uuid.uuid4().hex[:8]}', '-f', str(self.conf()), 'new-session', '-s', SOCKET,
                           '-c', self.cwd, 'sh', '-c', HOLD, 'ramabana', *self.command()], env

    def launch(self,
        prompt='',     # a one-shot question stays in this terminal
        as_json=False, # so does `--json`
        tmux='auto',   # `--tmux`; `off` stays out of tmux
    ):
        "Before an interactive session: ready tmux, and rerun inside ramabana's own server when outside tmux."
        if prompt or as_json or not self.isatty() or 'off' in (tmux, self._env('TMUX')): return
        if self.environ.get('TMUX'): return self.ext_keys()
        if self._env('WRAPPED') or not (self.tmux or self.offer_install()) or (self.version() or (0,)) < WRAP_MIN: return
        self.execvpe(*self.wrap())

A fake machine shows the relaunch. It has tmux 3.5 on `PATH`, and `execvpe` records what it would run:

In [ ]:
from types import SimpleNamespace
import tempfile

class Box:
    "Programs on PATH, the commands run, the prompts shown and the exec made."
    def __init__(self, have=('tmux',), version='tmux 3.5a', system='Darwin', answer='', ext='on'):
        self.have, self.version, self.system, self.answer, self.ext = set(have), version, system, answer, ext
        self.calls, self.asked, self.execd = [], [], None
    def which(self, name): return f'/bin/{name}' if name in self.have else None
    def run(self, cmd, **kw):
        self.calls.append(list(cmd))
        if 'install' in cmd: self.have.add('tmux')
        return SimpleNamespace(returncode=0, stdout={'-V': self.version, 'show': self.ext}.get(cmd[1], ''))
    def ask(self, q): return self.asked.append(q) or self.answer
    def execvpe(self, *a): self.execd = a

def mk(box, **kw):
    return Setup(tempfile.mkdtemp(), environ=kw.pop('environ', {'PATH': '/bin'}), argv=['-c', '--pane', 'on'], cwd='/work',
                 which=box.which, system=lambda: box.system, run=box.run, execvpe=box.execvpe, ask=box.ask, isatty=lambda: True)

box = Box()
mk(box).launch()
exe, argv, env = box.execd
test_eq((exe, env['RAMABANA_WRAPPED'], argv[argv.index('-c'):][:2], argv[-2:]), ('/bin/tmux', '1', ['-c', '/work'], ['--pane', 'on']))
argv[:12]

A one-shot question, `--json` and `--tmux off` stay in this terminal:

In [ ]:
for kw in (dict(prompt='hi'), dict(as_json=True), dict(tmux='off')):
    box = Box()
    mk(box).launch(**kw)
    test_eq(box.execd, None)

The `ramabana` script makes that choice before it imports the CLI, which takes 0.4s, so the process outside tmux execs fast. `launch_args` reads just enough of the command line to know the session is interactive. `refusal` holds the checks `main` makes before it starts anything, so a bad option is refused in this terminal and not on tmux's hold screen. A prompt, `--json`, `--doctor`, a refused option, or any option it is unsure of leaves the choice to `main`. When `run_cli` did launch, `LAUNCHED` stops `main` asking a second time.

In [ ]:
#| export
TMUX_MODES = {'auto': None, 'on': True, 'off': False}
PANE_MODES = ('auto', 'on', 'off')
PROFILES = ('auto', 'small', 'full')
PII_OFF = 'off'
PII_MODES = (PII_OFF, 'redact', 'refuse')

def refusal(tmux='auto', pane='auto', profile='auto', pii=PII_OFF, vault=False, python=False, attach='', agent_proxy=False,
            warm=False, no_warm=False):
    "Why `main` refuses these options before starting anything, or ''."
    if warm and no_warm: return '--warm and --no-warm contradict each other; pass one'
    if vault and (python or attach or agent_proxy): return 'there is no vault-backed host for a dhrishti session; drop --vault'
    for name, v, ok in (('tmux', tmux, TMUX_MODES), ('pane', pane, PANE_MODES), ('profile', profile, PROFILES), ('pii', pii, PII_MODES)):
        if v not in ok: return f"unknown --{name} {v!r}; choose one of {', '.join(ok)}"
    if pii != PII_OFF and not vault: return '--pii gates what a vault returns; add --vault'
    return ''

#: the `ramabana` options that take a value; every other option is a flag
VALUED = ('--root', '--model', '--approve', '--pii', '--theme', '--max-tool-calls', '--max-steps', '--cfg', '--resume',
          '--attach', '--tmux', '--pane', '--optin', '--profile')
#: options after which `main` returns before `launch`, or checks the python extra first
EARLY = ('-h', '--help', '--json', '--doctor', '--kernels', '--python', '--attach', '--agent-proxy')
#: the flags `refusal` reads
FLAGS = ('--vault', '--warm', '--no-warm')
LAUNCHED = False   # `run_cli` has run `launch`, so `main` does not ask again

def launch_args(argv):
    "`Setup`'s `cfg` and `launch`'s `tmux` for an interactive command line `main` accepts, read without its parser; None leaves it to `main`."
    kw, flags, it = dict(cfg=None, tmux='auto', pane='auto', profile='auto', pii=PII_OFF), set(), iter(argv)
    for a in it:
        k, eq, v = a.partition('=')
        if not a.startswith('-') or a == '-' or k in EARLY: return None
        if k in VALUED:
            if not eq and ((v := next(it, None)) is None or v.startswith('-')): return None
            kw[k[2:]] = v
        elif k in FLAGS: flags.add(k[2:].replace('-', '_'))
    if refusal(**{k: kw[k] for k in ('tmux', 'pane', 'profile', 'pii')}, **{f: True for f in flags}) or kw['tmux'] == 'off': return None
    return dict(cfg=kw['cfg'], tmux=kw['tmux'])

def run_cli():
    "The `ramabana` script: rerun inside tmux before anything heavy loads, then parse the command line for `cli.main`."
    global LAUNCHED
    if (msg := need('cli')): return _refuse(msg)()
    if (kw := launch_args(sys.argv[1:])) is not None:
        LAUNCHED = True
        Setup(kw['cfg']).launch(tmux=kw['tmux'])
    from fastcore.script import anno_parser
    from ramabana.cli import main
    args = vars(anno_parser(main.__wrapped__, pos=['prompt']).parse_args())
    for k in ('pdb', 'xtra'): args.pop(k, None)
    return main(**args)

In [ ]:
test_eq(launch_args(['--root', 'a,b', '--profile=full', '--no-web', '--cfg', '/c']), dict(cfg='/c', tmux='auto'))
for argv in (['hi'], ['--vault', 'hi'], ['-'], ['--json'], ['--doctor'], ['--tmux', 'off'], ['--pane', 'sideways']): test_eq(launch_args(argv), None)

## Installing tmux

When tmux is missing, `offer_install` asks once whether to run the command that installs it, and names it: `brew install tmux` on macOS, `sudo apt-get install -y tmux` or `sudo dnf install -y tmux` on Linux. The command runs attached to the terminal, so `sudo` can ask for a password. A no, or a yes whose install worked, goes in `<cfg>/setup.json`, and a no is never asked again. Ctrl+c at the prompt, or a failed or interrupted install, saves nothing, so the next run asks again. `ramabana --doctor` asks even so. Without tmux, the session starts in this terminal and one line says how to get the pane.

In [ ]:
#| export
@patch
def installer(self:Setup):
    "The command that installs tmux here, or None."
    if self.system() == 'Darwin': return ['brew', 'install', 'tmux'] if self.which('brew') else None
    pm = next((p for p in ('apt-get', 'dnf') if self.which(p)), None) if self.system() == 'Linux' else None
    return pm and [*(['sudo'] if self.which('sudo') else []), pm, 'install', '-y', 'tmux']

@patch
def _seen(self:Setup):
    try: return (self.cfg/'setup.json').read_json()
    except (OSError, ValueError): return {}

@patch
def _answer(self:Setup, cmd):
    "'yes' or 'no' to the install, or '' when ctrl+c or end of input left it unanswered."
    try: yes = self.ask(f'ramabana opens its now pane in tmux, which is missing. Run `{shlex.join(cmd)}`? [y/N] ').strip().lower() in ('y', 'yes')
    except (EOFError, KeyboardInterrupt): return ''
    return 'yes' if yes else 'no'

@patch
def _installed(self:Setup, cmd):
    try: return self.run(cmd).returncode == 0 and bool(self.tmux)
    except (OSError, KeyboardInterrupt): return False

@patch
def offer_install(self:Setup, force=False):
    "Ask once, or again when `force`, to install tmux: True when tmux is there now."
    if not force and 'install_tmux' in self._seen(): return False
    ans = 'none' if (cmd := self.installer()) is None else self._answer(cmd)
    if ans == 'yes' and not self._installed(cmd): ans = 'failed'
    if ans in ('none', 'no', 'yes'): (self.cfg/'setup.json').mk_write(json.dumps(self._seen() | {'install_tmux': ans}))
    if ans == 'yes': return True
    print(f'{WHY[ans]}; continuing without tmux · ramabana --doctor shows how to get the now pane', file=sys.stderr)
    return False

In [ ]:
box = Box(have={'brew'}, answer='y')
mk(box).launch()
test_eq((box.calls[0], box.execd is not None), (['brew', 'install', 'tmux'], True))
box.asked

In [ ]:
box = Box(have={'apt-get', 'sudo'}, system='Linux', answer='n')
s = mk(box)
s.launch()
s.launch()
test_eq((len(box.asked), box.execd, s._seen()), (1, None, {'install_tmux': 'no'}))

## Your own tmux

`ext_keys` reads `extended-keys` from the server it runs inside and turns it on only when it is `off`. Your `~/.tmux.conf` is never touched.

In [ ]:
#| export
@patch
def _show_ext(self:Setup):
    "This tmux server's `extended-keys`, or '' when tmux cannot say."
    if not (t := self.tmux): return ''
    r = self.run([t, 'show', '-sv', 'extended-keys'], capture_output=True, text=True)
    return '' if r.returncode else r.stdout.strip()

@patch
def ext_keys(self:Setup):
    "Turn on `extended-keys` in the tmux server we run inside when it is off: True if turned on."
    if self._show_ext() != 'off': return False
    self.run([self.tmux, 'set', '-s', 'extended-keys', 'on'], capture_output=True)
    print('turned on extended-keys in this tmux server so shift+enter reaches ramabana; ~/.tmux.conf is unchanged', file=sys.stderr)
    return True

In [ ]:
for ext, sets in (('off', True), ('on', False), ('always', False)):
    box = Box(ext=ext)
    mk(box, environ={'TMUX': '/tmp/tmux-501/default,1,0'}).launch()
    test_eq((['/bin/tmux', 'set', '-s', 'extended-keys', 'on'] in box.calls, box.execd), (sets, None))

## Whether the code index needs to run

A kosha sync reads the installed packages and the open roots. Before it starts, `index_gate` fingerprints both without importing kosha: the distributions `importlib.metadata` finds, with their mtimes, as kosha lists them; and each root's git HEAD and a digest of `git status` with the mtimes of the paths it names, or outside git the mtimes of the root's entries. When the fingerprint matches the one the last good sync saved in `<cfg>/kosha-state.json`, and each root's `.kosha/code.db` and kosha's env db exist, kosha is skipped. Either way `log_index` appends one line to `<cfg>/kosha.log`, and `--doctor` shows the last.

In [ ]:
#| export
KOSHA_STATE, KOSHA_LOG, LOG_MAX, LOG_KEEP = 'kosha-state.json', 'kosha.log', 64_000, 500

def kosha_env_db():
    "Where `Kosha()` keeps its package index: `<xdg data>/kosha/env.db`, as kosha computes it."
    from fastcore.xdg import xdg_data_home
    return xdg_data_home()/'kosha'/'env.db'

def pkg_state(dirs=None):
    "`{dist name-version: mtime}` for what `importlib.metadata` finds on `dirs`, default `sys.path`, as kosha lists packages."
    from importlib.metadata import distributions
    out = {}
    for d in distributions(**({'path': [str(p) for p in dirs]} if dirs else {})):
        if (p := getattr(d, '_path', None)) is None: continue
        try: out[re.sub(r'\.(dist|egg)-info$', '', p.name)] = p.stat().st_mtime
        except OSError: pass
    return out

def _worktree(root, out):
    "A digest of `git status --porcelain` and the mtimes of the paths it names, kosha's own `.kosha` left out."
    h, paths = hashlib.sha1(), [p for p in out.split('\0') if p and '.kosha' not in Path(p[3:]).parts]
    for p in paths:
        f = Path(root)/p[3:]
        h.update(p.encode()); h.update(repr(f.stat().st_mtime if f.exists() else 0).encode())
    return h.hexdigest()[:12]

def root_state(root):
    "A root's git HEAD and working-tree digest, staged changes included; outside git, the newest mtime of its entries."
    try:
        r = subprocess.run(['git', '-C', str(root), 'rev-parse', 'HEAD'], capture_output=True, text=True, timeout=10)
        st = subprocess.run(['git', '--no-optional-locks', '-C', str(root), 'status', '--porcelain=v1', '-z'], capture_output=True, text=True, timeout=10) if r.returncode == 0 else None
    except (OSError, subprocess.SubprocessError): r = None
    if r is not None and r.returncode == 0:
        return dict(head=r.stdout.strip(), worktree=_worktree(root, st.stdout))
    p = Path(root)
    return dict(mtime=max([0, *(c.lstat().st_mtime for c in p.iterdir() if c.name != '.kosha')]))

def index_state(roots, dirs=None):
    "The fingerprint of what a sync of `roots` reads."
    return dict(packages=pkg_state(dirs), roots={str(r): root_state(r) for r in roots})

def _pkg(name): return ' '.join(name.rpartition('-')[::2])

def index_changes(old, new, most=3):
    "Why `new` differs from `old`, e.g. `packages changed: +foo 1.2` or `leela HEAD a1b2→c3d4`; '' when it does not."
    if not old: return 'first sync'
    po, pn, why = old.get('packages', {}), new['packages'], []
    pk = [f'+{_pkg(n)}' for n in sorted(set(pn) - set(po))] + [f'-{_pkg(n)}' for n in sorted(set(po) - set(pn))] + \
         [f'~{_pkg(n)}' for n in sorted(set(pn) & set(po)) if pn[n] != po[n]]
    if pk: why.append('packages changed: ' + ' '.join(pk[:most]) + (f' and {len(pk)-most} more' if len(pk) > most else ''))
    for r, s in new['roots'].items():
        o, name = old.get('roots', {}).get(r), Path(r).name
        if o is None: why.append(f'{name} new root')
        elif s.get('head') != o.get('head'): why.append(f"{name} HEAD {(o.get('head') or '-')[:4]}→{(s.get('head') or '-')[:4]}")
        elif s.get('worktree') != o.get('worktree'): why.append(f'{name} worktree changed')
        elif s != o: why.append(f'{name} files changed')
    return '; '.join(why)

def index_gate(cfg, roots, dirs=None):
    "`(state, why)`: the fingerprint now, and why kosha must run, '' to skip it. A missing code or env db is a reason."
    try: old = json.loads((Path(cfg)/KOSHA_STATE).read_text())
    except (OSError, ValueError): old = None
    state = index_state(roots, dirs)
    missing = not kosha_env_db().exists() or any(not (Path(r)/'.kosha'/'code.db').exists() for r in roots)
    return state, '; '.join(w for w in (old and missing and 'index missing', index_changes(old, state)) if w)

def _replace(p, text):
    "Write `text` to `p` through a temp file and `os.replace`, so a reader never sees half of it."
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(f'.{p.name}.{os.getpid()}.tmp')
    tmp.write_text(text)
    os.replace(tmp, p)

def save_index(cfg, state):
    "Keep `state` as the last good sync's, beside the roots other sessions indexed."
    p = Path(cfg)/KOSHA_STATE
    try: old = json.loads(p.read_text())
    except (OSError, ValueError): old = {}
    _replace(p, json.dumps(dict(packages=state['packages'], roots={**old.get('roots', {}), **state['roots']})))

def log_index(cfg, decision, why, secs):
    "Append `when · run|skip · why · secs` to `<cfg>/kosha.log`, trimmed to its last `LOG_KEEP` lines past `LOG_MAX` bytes."
    p = Path(cfg)/KOSHA_LOG
    p.parent.mkdir(parents=True, exist_ok=True)
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} · {decision} · {why} · {secs:.2f}s\n"
    if p.exists() and p.stat().st_size > LOG_MAX: return _replace(p, ''.join((p.read_text().splitlines(True) + [line])[-LOG_KEEP:]))
    with open(p, 'a') as f: f.write(line)

def last_index(cfg):
    "The last line of `<cfg>/kosha.log`, or ''."
    try: return (Path(cfg)/KOSHA_LOG).read_text().splitlines()[-1]
    except (OSError, IndexError): return ''

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as d:
    d = Path(d); (site := d/'site').mkdir(); (site/'fastcore-1.8.0.dist-info').mkdir(); (root := d/'proj').mkdir()
    state, why = index_gate(d/'cfg', [root], [site])
    test_eq((list(state['packages']), why), (['fastcore-1.8.0'], 'first sync'))
    save_index(d/'cfg', state)
    test_eq(index_gate(d/'cfg', [root], [site])[1], 'index missing')
    (root/'.kosha').mkdir(); (root/'.kosha'/'code.db').write_text('')
    if kosha_env_db().exists(): test_eq(index_gate(d/'cfg', [root], [site])[1], '')
    (site/'foo-1.2.dist-info').mkdir()
    assert index_gate(d/'cfg', [root], [site])[1].endswith('packages changed: +foo 1.2')
    log_index(d/'cfg', 'skip', 'unchanged', .01)
    test_eq(last_index(d/'cfg').split(' · ')[1:], ['skip', 'unchanged', '0.01s'])

## `--doctor`

`ramabana --doctor` prints one line for each thing the pane and the keys depend on, and the code index's last run-or-skip, then offers the install when tmux is missing, even after an earlier no.

In [ ]:
#| export
@patch
def doctor(self:Setup):
    "Print one line per check, offer to install tmux when it is missing, and return 0."
    v, about, conf, pane = self.version(), self.about(), self.cfg/'tmux.conf', self.which('ramabana-pane')
    keys = f'{self._show_ext() or "unknown"} in this tmux' if self.environ.get('TMUX') else \
           f"{'on' if v and v >= KEYS_MIN else 'needs tmux 3.2+'} in ramabana's own tmux server"
    miss = [x for x in EXTRAS if need(x)]
    split = f'ok on {about}' if v and v >= SPLIT_MIN else 'needs tmux 3.1+' + (f', this is {about}; /pane prints a command to run instead' if v else '')
    print(f'tmux: {about} at {self.tmux}' if v else 'tmux: not found', f'split: {split}', f'extended-keys: {keys}',
          f'config: {conf}' + ('' if conf.exists() else ', written by the first session'),
          f'ramabana-pane: {pane}' if pane else 'ramabana-pane: not on PATH; the pane runs python -m ramabana.pane',
          f"extras: {', '.join(miss)} missing: pip install 'ramabana[{','.join(miss)}]'" if miss else 'extras: all installed',
          f'kosha: {last_index(self.cfg) or f"no decision yet; the first session writes {self.cfg/KOSHA_LOG}"}', sep='\n')
    if not v: self.offer_install(force=True)
    return 0

In [ ]:
mk(Box(have={'tmux', 'ramabana-pane'}, version='tmux 3.0a')).doctor()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()